# HM-GAT inference

This notebook demonstrates inference with the pretrained **HM-GAT**
model for predicting experimental UV–Vis absorption spectra of organic
molecule–solvent pairs.

## Input

- Molecular SMILES
- Solvent SMILES

## Output

- Predicted extinction coefficients from 330 to 1000 nm at 2 nm resolution

The model uses 2D molecular and solvent structures only and does not
require optimized 3D geometries or DFT-calculated spectra.


In [ ]:
# ============================================================
# Cell 0. HM-GAT public inference setup
# ============================================================

import os
import sys
import yaml
import tempfile
import importlib.util
import contextlib
import io
import warnings
from pathlib import Path
from collections import OrderedDict

import numpy as np
import pandas as pd

import torch
from torch.utils.data import DataLoader

import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=UserWarning)

# ------------------------------------------------------------
# User setting
# ------------------------------------------------------------

VERBOSE = False

# ------------------------------------------------------------
# Package root
# ------------------------------------------------------------

def find_package_root(start_path):
    """
    Locate the HM-GAT package root using required public-package markers.
    """

    start_path = Path(start_path).resolve()

    candidate_paths = [
        start_path,
        start_path / "HM_GAT",
        *start_path.parents,
    ]

    checked_paths = set()

    for candidate in candidate_paths:
        candidate = Path(candidate).resolve()

        if candidate in checked_paths:
            continue

        checked_paths.add(candidate)

        required_markers = [
            candidate / "model" / "network.py",
            candidate / "model" / "HM_GAT_pretrained.pth",
            candidate / "resources" / "functional_group.csv",
            candidate / "D4CMPP2",
        ]

        if all(path.exists() for path in required_markers):
            return candidate

    raise FileNotFoundError(
        "Could not locate the HM-GAT package root. "
        "Open this notebook from the cloned repository or set the "
        "working directory to the repository root."
    )


PACKAGE_ROOT = find_package_root(
    Path.cwd()
)

os.chdir(
    PACKAGE_ROOT
)

MODEL_DIR = PACKAGE_ROOT / "model"
DATA_DIR = PACKAGE_ROOT / "data"
RESOURCE_DIR = PACKAGE_ROOT / "resources"
TMP_RUNTIME_DIR = PACKAGE_ROOT / "_tmp_runtime"
GRAPH_DIR = TMP_RUNTIME_DIR / "graph_cache"
LOCAL_D4CMPP2_DIR = PACKAGE_ROOT / "D4CMPP2"


TMP_RUNTIME_DIR.mkdir(parents=True, exist_ok=True)
GRAPH_DIR.mkdir(parents=True, exist_ok=True)

# Local package import priority
sys.path.insert(0, str(PACKAGE_ROOT))

# ------------------------------------------------------------
# Final model setting
# ------------------------------------------------------------

FINAL_ALIAS = "HM_GAT"
FINAL_MODEL_LABEL = "HM-GAT"

FINAL_MODEL_ID = "HM_GAT_public"

FINAL_SCULPTOR_INDEX = (9, 2, 1)
FINAL_GRAPH_TAG = "img921"
FINAL_SPEC_SPLIT_DIMS = (48, 86, 202)

DATA_NAME = "test_dataset"

CSV_PATH = DATA_DIR / "test_dataset.csv"
FUNCTIONAL_GROUP_CSV = RESOURCE_DIR / "functional_group.csv"

FINAL_NETWORK_PATH = MODEL_DIR / "network.py"
FINAL_CONFIG_PATH = MODEL_DIR / "config.yaml"
FINAL_WEIGHT_PATH = MODEL_DIR / "HM_GAT_pretrained.pth"

# ------------------------------------------------------------
# Spectrum setting
# ------------------------------------------------------------

WAVELENGTHS = np.arange(330, 1002, 2, dtype=float)
TARGET_COLS = [str(int(w)) for w in WAVELENGTHS]

assert len(WAVELENGTHS) == 336
assert len(TARGET_COLS) == 336

# ------------------------------------------------------------
# Runtime
# ------------------------------------------------------------

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

BATCH_SIZE = 256
NUM_WORKERS = 0
PIN_MEMORY = False

# ------------------------------------------------------------
# Quiet context
# ------------------------------------------------------------

@contextlib.contextmanager
def quiet_if_needed(verbose=False):
    if verbose:
        yield
    else:
        with contextlib.redirect_stdout(io.StringIO()):
            yield

print("[Cell 0] Config ready")
print("PACKAGE_ROOT:", PACKAGE_ROOT)
print("DEVICE      :", DEVICE)

In [ ]:
# ============================================================
# Cell 1. Public resource check
# ============================================================

required_paths = {
    "network.py": FINAL_NETWORK_PATH,
    "config.yaml": FINAL_CONFIG_PATH,
    "HM_GAT_pretrained.pth": FINAL_WEIGHT_PATH,
    "test_dataset.csv": CSV_PATH,
    "functional_group.csv": FUNCTIONAL_GROUP_CSV,
    "D4CMPP2": LOCAL_D4CMPP2_DIR,
}

missing_resources = [
    name
    for name, path in required_paths.items()
    if not Path(path).exists()
]

if missing_resources:
    raise FileNotFoundError(
        f"Missing required public resources: {missing_resources}"
    )


from D4CMPP2.src.DataManager.ISADataManager import ISADataManager


public_data_head = pd.read_csv(
    CSV_PATH,
    nrows=5,
)

public_columns = set(
    public_data_head.columns
)

assert "compound" in public_columns, (
    "test_dataset.csv must contain a 'compound' column."
)

assert "solvent" in public_columns, (
    "test_dataset.csv must contain a 'solvent' column."
)

missing_target_columns = [
    column
    for column in TARGET_COLS
    if (
        column not in public_columns
        and str(column) not in public_columns
    )
]

assert not missing_target_columns, (
    "test_dataset.csv is missing wavelength columns: "
    f"{missing_target_columns[:10]}"
)

print("[Cell 1] Public resource check passed")
print("Package root       :", PACKAGE_ROOT)
print("Public test dataset:", CSV_PATH)
print("Pretrained weight  :", FINAL_WEIGHT_PATH)


In [ ]:
# ============================================================
# Cell 2. Load pretrained HM-GAT model
# ============================================================

class TupleSafeLoader(yaml.SafeLoader):
    pass


def construct_python_tuple(loader, node):
    return tuple(loader.construct_sequence(node))


TupleSafeLoader.add_constructor(
    "tag:yaml.org,2002:python/tuple",
    construct_python_tuple,
)


def load_saved_config(config_path):
    with open(config_path, "r") as f:
        return dict(yaml.load(f, Loader=TupleSafeLoader))


def import_network_from_path(network_py):
    spec = importlib.util.spec_from_file_location("local_hm_gat_network", str(network_py))
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)

    assert hasattr(module, "network"), "network.py must contain class/function named 'network'."
    return module.network


def clean_state_dict_keys(raw):
    if isinstance(raw, dict):
        if "state_dict" in raw and isinstance(raw["state_dict"], dict):
            raw = raw["state_dict"]
        elif "model_state_dict" in raw and isinstance(raw["model_state_dict"], dict):
            raw = raw["model_state_dict"]

    new_sd = OrderedDict()

    for k, v in raw.items():
        new_k = k
        for prefix in ["module.", "model.", "network.", "net."]:
            if new_k.startswith(prefix):
                new_k = new_k[len(prefix):]
        new_sd[new_k] = v

    return new_sd


# ------------------------------------------------------------
# Model config
# ------------------------------------------------------------

model_config = load_saved_config(FINAL_CONFIG_PATH)

# Training-only paths must not be used during public inference.
model_config["encoder_pretrain_path"] = None
model_config["pretrained_path"] = None
model_config["checkpoint_path"] = None


model_config["target_dim"] = 336
model_config["output_dim"] = 336
model_config["spec_split_dims"] = FINAL_SPEC_SPLIT_DIMS

model_config["sculptor_index"] = FINAL_SCULPTOR_INDEX
model_config["sculptor_s"] = FINAL_SCULPTOR_INDEX[0]
model_config["sculptor_c"] = FINAL_SCULPTOR_INDEX[1]
model_config["sculptor_a"] = FINAL_SCULPTOR_INDEX[2]

# PM6-free model
model_config["numeric_input_columns"] = []
model_config["use_pm6_spectrum"] = False
model_config["query_source"] = "graph_mol_feat"

# Safe defaults
model_config.setdefault("node_dim", 45)
model_config.setdefault("edge_dim", 1)
model_config.setdefault("d_edge_dim", 4)

model_config.setdefault("linear_layers", 3)
model_config.setdefault("hidden_dim", 128)
model_config.setdefault("dropout", 0.1)
model_config.setdefault("gat_dropout", 0.0)
model_config.setdefault("conv_layers", 6)
model_config.setdefault("heads", 8)

model_config.setdefault("i2i_residual", True)
model_config.setdefault("d2score_mode", "vector_gate")
model_config.setdefault("use_split_final_linear", False)
model_config.setdefault("mlp_activation", "leakyrelu")

model_config.setdefault("spectral_correction_mode", "conv1d_local")
model_config.setdefault("conv_correction_channels", 16)
model_config.setdefault("conv_kernel_size", 7)
model_config.setdefault("use_learnable_conv_scale", True)
model_config.setdefault("conv_correction_scale_init", 0.1)

# ------------------------------------------------------------
# Load model
# ------------------------------------------------------------

NetClass = import_network_from_path(FINAL_NETWORK_PATH)

with quiet_if_needed(VERBOSE):
    model = NetClass(model_config)

raw = torch.load(FINAL_WEIGHT_PATH, map_location="cpu")
state_dict = clean_state_dict_keys(raw)

missing, unexpected = model.load_state_dict(state_dict, strict=False)

assert len(missing) == 0, f"Missing keys: {missing[:20]}"
assert len(unexpected) == 0, f"Unexpected keys: {unexpected[:20]}"

model = model.to(DEVICE)
model.eval()

print("[Cell 2] Model loaded")
print("Model :", FINAL_ALIAS)
print("Weight:", FINAL_WEIGHT_PATH.name)
print("Device:", DEVICE)


In [ ]:
# ============================================================
# Cell 3. DataManager builder
# ============================================================

def make_base_datamanager_config(
    data_name,
    csv_path,
    graph_dir,
    model_path,
    batch_size=1,
):
    s, c, a = FINAL_SCULPTOR_INDEX

    return {
        "data": data_name,
        "DATA_PATH": str(csv_path),
        "GRAPH_DIR": str(graph_dir),
        "MODEL_PATH": str(model_path),

        "target": TARGET_COLS,
        "molecule_columns": ["compound", "solvent"],

        # PM6-free inference
        "numeric_input_columns": [],

        "explicit_h_columns": ["solvent"],

        "scaler": "identity",
        "split_random_seed": 42,

        "batch_size": batch_size,
        "shuffle": False,
        "num_workers": NUM_WORKERS,
        "pin_memory": PIN_MEMORY,

        "sculptor_s": s,
        "sculptor_c": c,
        "sculptor_a": a,

        "hidden_dim": 128,
        "conv_layers": 6,
        "dropout": 0.1,
        "gat_dropout": 0.0,
        "heads": 8,
        "linear_layers": 3,
    }


def build_loader_from_dataset(dm_obj, dataset, batch_size=1):
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        collate_fn=dm_obj.dataset.collate,
        pin_memory=False,
        drop_last=False,
    )


print("[Cell 3] DataManager helper ready")


In [ ]:
# ============================================================
# Cell 4. Prediction core
# ============================================================

def adapt_unwrapped_for_hm_gat(unwrapped):
    u = dict(unwrapped)

    if "compound_graphs" in u:
        u.setdefault("compound_graph", u["compound_graphs"])
        u.setdefault("graph", u["compound_graphs"])

    if "solvent_graphs" in u:
        u.setdefault("solvent_graph", u["solvent_graphs"])
        u.setdefault("solv_graphs", u["solvent_graphs"])
        u.setdefault("solv_graph", u["solvent_graphs"])

    if "compound_r_node" in u:
        u.setdefault("compound_node_feature", u["compound_r_node"])
        u.setdefault("compound_node_feats", u["compound_r_node"])

    if "compound_r2r_edge" in u:
        u.setdefault("compound_edge_feature", u["compound_r2r_edge"])
        u.setdefault("compound_edge_feats", u["compound_r2r_edge"])

    if "solvent_r_node" in u:
        u.setdefault("solvent_node_feature", u["solvent_r_node"])
        u.setdefault("solvent_node_feats", u["solvent_r_node"])
        u.setdefault("solv_node_feature", u["solvent_r_node"])
        u.setdefault("solv_node_feats", u["solvent_r_node"])

    if "solvent_r2r_edge" in u:
        u.setdefault("solvent_edge_feature", u["solvent_r2r_edge"])
        u.setdefault("solvent_edge_feats", u["solvent_r2r_edge"])
        u.setdefault("solv_edge_feature", u["solvent_r2r_edge"])
        u.setdefault("solv_edge_feats", u["solvent_r2r_edge"])

    return u


def predict_loader(model, loader, dm_obj, device=DEVICE):
    preds = []

    model.eval()

    with torch.no_grad():
        for batch in loader:
            unwrapped = dm_obj.unwrapper(device=str(device), **batch)
            adapted = adapt_unwrapped_for_hm_gat(unwrapped)

            y_hat = model(**adapted)

            assert y_hat.ndim == 2
            assert y_hat.shape[1] == 336
            assert torch.isfinite(y_hat).all(), "Prediction contains non-finite values."

            preds.append(y_hat.detach().cpu())

    return torch.cat(preds, dim=0).numpy()


print("[Cell 4] Prediction core ready")


In [ ]:
# ============================================================
# Cell 5. Single SMILES pair prediction function
# ============================================================

def make_single_pair_dataframe(compound_smiles, solvent_smiles):
    """
    ISADataManager.split_data() requires train/val/test rows.
    Therefore, the same input pair is duplicated into 3 rows.
    Prediction is taken from the test row.
    """
    rows = []

    for set_name in ["train", "val", "test"]:
        row = {
            "compound": compound_smiles,
            "solvent": solvent_smiles,
            "set": set_name,
        }

        # Dummy target values. These are not used as input.
        for c in TARGET_COLS:
            row[c] = 0.0

        rows.append(row)

    return pd.DataFrame(rows)


def predict_single_pair(
    compound_smiles,
    solvent_smiles,
    request_name="single_prediction",
    plot=True,
):
    """
    Predict 330–1000 nm absorption spectrum for one compound-solvent pair.

    Returns
    -------
    spectrum_df:
        Wavelength_nm, Predicted

    info:
        Prediction metadata.
    """

    with tempfile.TemporaryDirectory(prefix="hm_gat_", dir=str(TMP_RUNTIME_DIR)) as tmp_root:
        tmp_root = Path(tmp_root)
        input_dir = tmp_root / "input"
        graph_dir = tmp_root / "graphs"

        input_dir.mkdir(exist_ok=True)
        graph_dir.mkdir(exist_ok=True)

        input_df = make_single_pair_dataframe(compound_smiles, solvent_smiles)
        input_csv = input_dir / f"{request_name}.csv"
        input_df.to_csv(input_csv, index=False)

        single_config = make_base_datamanager_config(
            data_name=request_name,
            csv_path=input_csv,
            graph_dir=graph_dir,
            model_path=RESOURCE_DIR,
            batch_size=1,
        )

        with quiet_if_needed(VERBOSE):
            dm_single = ISADataManager(single_config)
            dm_single.init_data()
            dm_single.prepare_dataset()
            dm_single.split_data()

        assert len(dm_single.test_dataset) == 1

        test_loader_single = build_loader_from_dataset(
            dm_obj=dm_single,
            dataset=dm_single.test_dataset,
            batch_size=1,
        )

        pred = predict_loader(
            model=model,
            loader=test_loader_single,
            dm_obj=dm_single,
            device=DEVICE,
        )

    predicted = pred[0]

    spectrum_df = pd.DataFrame({
        "Wavelength_nm": WAVELENGTHS,
        "Predicted": predicted,
    })

    lambda_abs_pred_nm = float(WAVELENGTHS[np.nanargmax(predicted)])
    max_predicted_value = float(np.nanmax(predicted))

    info = {
        "model_alias": FINAL_ALIAS,
        "model_label": FINAL_MODEL_LABEL,
        "compound_smiles": compound_smiles,
        "solvent_smiles": solvent_smiles,
        "lambda_abs_pred_nm": lambda_abs_pred_nm,
        "max_predicted_value": max_predicted_value,
    }

    print("[Prediction complete]")
    print("λabs_pred:", f"{lambda_abs_pred_nm:.0f} nm")
    print("max_pred :", f"{max_predicted_value:.4f}")

    if plot:
        plt.figure(figsize=(7, 4))
        plt.plot(spectrum_df["Wavelength_nm"], spectrum_df["Predicted"])
        plt.xlabel("Wavelength (nm)")
        plt.ylabel("Predicted intensity")
        plt.title("Predicted UV–Vis absorption spectrum")
        plt.grid(alpha=0.3)
        plt.tight_layout()
        plt.show()

    return spectrum_df, info


print("[Cell 5] Single prediction function ready")


In [ ]:
# ============================================================
# Cell 6. Public inference example
# ============================================================

EXAMPLE_ROW_INDEX = 0

public_test_df = pd.read_csv(
    CSV_PATH
)

if not 0 <= EXAMPLE_ROW_INDEX < len(public_test_df):
    raise IndexError(
        f"EXAMPLE_ROW_INDEX must be between 0 and "
        f"{len(public_test_df) - 1}."
    )

example_row = public_test_df.iloc[
    EXAMPLE_ROW_INDEX
]

example_compound_smiles = str(
    example_row["compound"]
)

example_solvent_smiles = str(
    example_row["solvent"]
)


spectrum_df, prediction_info = predict_single_pair(
    compound_smiles=example_compound_smiles,
    solvent_smiles=example_solvent_smiles,
    request_name=f"public_example_{EXAMPLE_ROW_INDEX:03d}",
    plot=False,
)


resolved_example_target_columns = []

for target_column in TARGET_COLS:
    if target_column in public_test_df.columns:
        resolved_example_target_columns.append(
            target_column
        )

    elif str(target_column) in public_test_df.columns:
        resolved_example_target_columns.append(
            str(target_column)
        )

    else:
        raise KeyError(
            f"Missing wavelength column: {target_column}"
        )


experimental_spectrum = pd.to_numeric(
    example_row[
        resolved_example_target_columns
    ],
    errors="coerce",
).to_numpy(dtype=float)

wavelengths_example = np.asarray(
    [
        float(column)
        for column in resolved_example_target_columns
    ],
    dtype=float,
)

predicted_spectrum = spectrum_df[
    "Predicted"
].to_numpy(dtype=float)


if len(predicted_spectrum) != len(wavelengths_example):
    raise RuntimeError(
        "Prediction and wavelength dimensions do not match."
    )


print("=" * 80)
print("[Public HM-GAT example]")
print("=" * 80)
print("Dataset row index :", EXAMPLE_ROW_INDEX)
print("Compound SMILES   :", example_compound_smiles)
print("Solvent SMILES    :", example_solvent_smiles)
print("Prediction points :", len(predicted_spectrum))
print("=" * 80)


plt.figure(
    figsize=(8.5, 5.2),
    dpi=140,
)

plt.plot(
    wavelengths_example,
    experimental_spectrum,
    linewidth=2.0,
    label="Experimental",
)

plt.plot(
    wavelengths_example,
    predicted_spectrum,
    linewidth=1.8,
    label="HM-GAT prediction",
)

plt.xlabel(
    "Wavelength (nm)"
)

plt.ylabel(
    "Extinction coefficient"
)

plt.xlim(
    wavelengths_example.min(),
    wavelengths_example.max(),
)

plt.legend(
    frameon=False
)

plt.grid(
    True,
    alpha=0.2,
)

plt.tight_layout()
plt.show()


display(
    spectrum_df.head()
)
